# Getting images and MS COCO captions

This notebook contains code to do the following:

* Identifying the 1K images for which we have fMRI responses
* Downloading these images
* Mapping these images to the MS COCO images
* Getting the MS COCO annotations we may need for these images

In [1]:
from scipy.io import loadmat
import numpy as np
from src.paths import ROOT
from src.utils import open_json, dict_to_json
import os.path
import pandas as pd
import json
import os
import urllib

Let's first load a file containing identifiers for the 'special set' (`nsd_expdesign.mat`, available [here](/Users/anna/Downloads/nsd_expdesign.mat)).


In [7]:
exp_data = loadmat(ROOT / 'data/images/nsd_expdesign.mat', simplify_cells = True)
exp_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'basiccnt', 'masterordering', 'sharedix', 'stimpattern', 'subjectim'])

In [8]:
print(exp_data['sharedix'].shape)
print(exp_data['sharedix'][:10])

(1000,)
[2951 2991 3050 3078 3147 3158 3165 3172 3182 3387]


Images are available in [this folder](https://natural-scenes-dataset.s3.amazonaws.com/index.html#nsddata/stimuli/nsd/shared1000/) (`nsddata/stimuli/nsd/shared1000/`). We need a couple of helper functions to map image IDs to the actual file names we need to download.

In [9]:
def pad_shared_id(index):

    str_ind = str(index)
    len_pad_needed = 4 - len(str_ind)
    padded_idx = f"{'0'*len_pad_needed}{str_ind}"

    return padded_idx

def pad_id(index):

    str_ind = str(index)
    len_pad_needed = 5 - len(str_ind)
    padded_idx = f"{'0'*len_pad_needed}{str_ind}"

    return padded_idx

pad_shared_id(1)

'0001'

In [ ]:
# downloading images

base_url = "https://natural-scenes-dataset.s3.amazonaws.com/nsddata/stimuli/nsd/shared1000/"
for i, ind in enumerate(exp_data['sharedix'][0]):

    img_name = f"shared{pad_shared_id(i+1)}_nsd{pad_id(ind)}.png"
    complete_url = base_url + img_name
    if not os.path.isfile(ROOT / "data/imagescoco_images" / img_name):
        urllib.request.urlretrieve(complete_url, ROOT / "data/images/stimuli" / img_name)


In [6]:
# checking all images are there

len(os.listdir(ROOT / "data/images/stimuli"))

1000

## Explore MS COCO annotations

Awesome—we have all images! Now, let's try to map them to the relevant MS COCO annotations, stored in the folder `data/images/original_coco_annotations`.

In [2]:
# we may need this file to do the mapping
df = pd.read_csv(ROOT / "data/images/nsd_stim_info_merged.csv")

In [8]:
df.head(2)

,Unnamed: 0,cocoId,cocoSplit,cropBox,loss,nsdId,flagged,BOLD5000,shared1000,subject1,...,subject5_rep2,subject6_rep0,subject6_rep1,subject6_rep2,subject7_rep0,subject7_rep1,subject7_rep2,subject8_rep0,subject8_rep1,subject8_rep2
0,0,532481,val2017,"(0, 0, 0.1671875, 0.1671875)",0.1,0,False,False,False,0,...,0,0,0,0,0,0,0,0,0,0
1,1,245764,val2017,"(0, 0, 0.125, 0.125)",0.0,1,False,False,False,0,...,0,0,0,0,13985,14176,28603,0,0,0


In [3]:
# let's filter and keep only the images included in shared1K

df_sub = df.loc[df.shared1000==True]
df_sub.shape

(1000, 41)

In [10]:
df_sub.head(2)

,Unnamed: 0,cocoId,cocoSplit,cropBox,loss,nsdId,flagged,BOLD5000,shared1000,subject1,...,subject5_rep2,subject6_rep0,subject6_rep1,subject6_rep2,subject7_rep0,subject7_rep1,subject7_rep2,subject8_rep0,subject8_rep1,subject8_rep2
2950,2950,262145,train2017,"(0, 0, 0.16640625, 0.16640625)",0.09375,2950,False,True,True,1,...,27566,2616,9716,27566,2616,9716,27566,2616,9716,27566
2990,2990,262239,train2017,"(0, 0, 0.1671875, 0.1671875)",0.10000,2990,False,True,True,1,...,27711,18458,18697,27711,18458,18697,27711,18458,18697,27711


In [10]:
# this indexing is not the same as the exp_data file, so we need to correct
coco_to_nsd = {cocoid: nsd_id+1 for cocoid, nsd_id in zip(df_sub.cocoId, df_sub.nsdId)}

Looking at the MS Coco annotations

In [4]:
train_captions = open_json(ROOT / "data/images/original_coco_annotations/captions_train2017.json")

In [20]:
train_captions['annotations'][0]

{'image_id': 203564,
 'id': 37,
 'caption': 'A bicycle replica with a clock as the front wheel.'}

In [28]:
shared_1k_captions = {}
for annot in train_captions['annotations']:
    if annot['image_id'] in coco_to_nsd:
        # print('got here')
        if coco_to_nsd[annot['image_id']] not in shared_1k_captions:
            shared_1k_captions[coco_to_nsd[annot['image_id']]] = []
        shared_1k_captions[coco_to_nsd[annot['image_id']]].append(annot['caption'])
        


In [29]:
# let's do the same for the validation captions
val_captions = open_json(ROOT / "data/images/original_coco_annotations/captions_val2017.json")

In [30]:
# shared_1k_captions = {}
for annot in val_captions['annotations']:
    if annot['image_id'] in coco_to_nsd:
        if coco_to_nsd[annot['image_id']] not in shared_1k_captions:
            print('Found a new caption')
            shared_1k_captions[coco_to_nsd[annot['image_id']]] = []
        shared_1k_captions[coco_to_nsd[annot['image_id']]].append(annot['caption'])
    

Great! This means that we can consider only the train split for the other annotations. Now, let's save these filtered annotations.

In [32]:
dict_to_json(shared_1k_captions, ROOT / "data/images/cleaned_coco_annotations/captions.json")